# Medical imaging

## Practicum 1: Computed Tomography Image Visualisation & Reconstruction
Oct-Nov 2026
==============================================================================================

In this practicum, you will work with several topics covered in the theory sessions (Block 1). In particular you will (1) simulate a simple and more complex phantoms, (2) apply Radom forward transformation to simulate the computed tomography (CT) projections over a single slice (i.e., sinograms), and (3) reconstruct the original phantom intensity values using different image reconstruction algorithms. In addition, the Shepp–Logan (head) phantom will be used to analyse noise during reconstruction.

### Aims:
- Generate simple and complex phantoms.
- Understand the principles of CT image reconstruction using a fan beam geometry with the aid of ``skimage``.
- Be able to generate and understand image projections (i.e., sinograms).
- Reconstruct slices from simple and more complex objects (i.e., Shepp–Logan phantom) using back projection.
- Analyse and compare reconstructed data with ground truth data.
- Observe the difference between ideal (noise free) and noisy image reconstruction.

``Remember to comment your code and provide some discussion on the results obtained for each section.``



In [9]:
%matplotlib inline

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon

### 1.1 Visualise a lung CT dicom image and create a phantom (2 points)

Open and visualise the Lung CT volume stored at the `LungCT-Diagnosis` accompanied folder. You can use mango viewer, ITK-Snap, etc. Then, using python commands, read each of the indivdual `dcm` images, and display some 2D slices. Some basic image processing might be needed to adjust the contrast/windows level for better visualisation of all tissues. Then, create a 3D phantom with the ribcages based on their HU (e.g., thresholding) and visualise the 3D rendering. Congratulations, you have created your first anthropomorphic phantom!

<img src='images_4_notebook/LungPhantomImage.png' width="400">


<img src='images_4_notebook/LungPhantomRendering.png' width="400">


Data was taken from https://www.cancerimagingarchive.net/collection/lungct-diagnosis/ 


In [2]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pydicom as pdc
from pydicom.filereader import dcmread
from skimage.exposure import rescale_intensity
from skimage.morphology import ball, binary_opening, binary_closing, remove_small_objects
from skimage.measure import marching_cubes
from scipy.ndimage import gaussian_filter
import plotly.graph_objects as go

In [3]:
# ---------- 1) Read DICOM series ----------
DATA_DIR = Path("LungCT-Diagnosis")  # set to your dataset root

def read_series(dicom_dir: Path):
    files = sorted([p for p in dicom_dir.rglob("*.dcm")])
    assert files, f"No DICOMs under {dicom_dir}"
    # read all headers to sort by position or instance
    metas = []
    for f in files:
        ds = dcmread(str(f), stop_before_pixels=True, force=True)
        z = None
        if hasattr(ds, "ImagePositionPatient"):
            z = float(ds.ImagePositionPatient[2])
        inst = int(getattr(ds, "InstanceNumber", 0))
        metas.append((f, z, inst))
    # prefer ImagePositionPatient for sort, else InstanceNumber
    if all(m[1] is not None for m in metas):
        metas.sort(key=lambda x: x[1])
    else:
        metas.sort(key=lambda x: x[2])


# insert your code here


In [4]:
# ---------- 2) 2D visualisation with window/level ----------

# insert your code here


In [29]:
# ---------- 3) Ribcage phantom via HU threshold ----------
# Simple bone mask. Typical cortical bone > 300 HU.
BONE_TH = 100.0
bone_mask = vol_hu >= BONE_TH

# insert your code here


Bone voxels: 371037


In [5]:
# ---------- 4) 3D rendering with marching cubes ----------
# Marching cubes expects z-y-x ordering and voxel spacing.


# insert your code here


### 1.2 Create a simple and homogeneous phantom (1.5 points)

Now create from scratch a more simple 2D phantom which contains a fake (bright) lesion as shown below:

<img src='images_4_notebook/PhantomImage.png' width="400">

- Phantom size: 256 x 256 pixels.
- Phantom must contain 3 tissues:
    - Background = 0 pixel intensity value
    - Tissue 1 = 1 (radius = 100 pixels)
    - Tissue 2 (lesion) = 2 (radius = 5 pixels)
    
Help: To create the circunferences, you can use the function ``create_circular_mask``, or define your own function. The lesion can be placed anywhere within the phantom, except at the exact center.

Once constructed, add a second lesion in the same row (same size and pixel intensity), plot the original phantom and a pixel intensity (horizontal) profile along the synthetic lesions, similarly as shown above (but with two lesions). Always remember to include the units of the axis when ploting a graph.

``Questions``: What challenges did you face? How did you deal with them?

In [1]:
def create_circular_mask(h, w, center=None, radius=None):

    if center is None: # use the middle of the image
        center = (int(w/2), int(h/2))
    if radius is None: # use the smallest distance between the center and image walls
        radius = min(center[0], center[1], w-center[0], h-center[1])

    Y, X = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((X - center[0])**2 + (Y-center[1])**2)

    mask = dist_from_center <= radius
    return mask
# https://newbedev.com/how-can-i-create-a-circular-mask-for-a-numpy-array

In [1]:
# insert your code here



### 1.3 Create the projections of the phantom (2 points)

Generate the projections of the previously generated 2D phantom using the ``radon`` transform from ``skimage.transform``. Play with the different parameters to optimise sinogram. Then, show the sinogram of the projections.

In this section, considered the following approch:

Use different number of projections (i.e. 4, 20, 50, 100, 200, 300, 360) considering that in all cases the simulated projections cover the 360º. For example, 100 projections are taken in the range from 0º to 360º.


``Questions``: How do the sinograms changes with number of projections? What is the effect of increasing/decrasing the number of projections? 

In [3]:
# insert your code here


### 1.4 Reconstruction with Filtered Back Projection (FBP) (2.5 points)

FBP is one of the most simple reconstruction methods to reconstruct images in CT. Explore the ``iradon`` from ``skimage.transform`` using the different filters available (Ramp filter (default), Shepp-logan, Cosine, Hamming, Hann). 

Make use of the ``matplotlib`` to show the original and reconstructed images of the phantom and compare the pixel intensity signal across the lesion profile (similarly to section 2.1).In addition, provide evaluation metric you could consider useful for this purpose (image difference, mean square error (MSE), peak signal to noise ratio (PSNR), structural index similarity (SSIM), etc.). Then, discuss the results.

In [6]:
# insert your code here


### 1.5 Sheep logan phantom (2 points)

So far we have used a quite simple phantom with only 2 tissues. Now, repeat the prior analysis for the **noisy sinogram** for one case (approach, filter,...) using a (slightly) more advanced head test object, the Sheep-logan phantom. Do you observe particular differences between the reconstrucction results from both phantoms? 


<img src='images_4_notebook/SheepLoganPhantom.png' width="1000">



In order to create **noisy sinograms**, you can add Poison noise (``np.random.poisson``; perhaps with lam = 10) to your noise-free sinograms (not to the image!) and reconstruct the phantom images with the different filters. Plot the reconstruction image and the (horizontal) intensity profile along specific row. Then, comment on the effects of the filters on the reconstructed phatom images.


In [31]:
from skimage.data import shepp_logan_phantom

logan_phantom = shepp_logan_phantom()


# insert your code here



###  Generative AI Declaration Policy
If generative AI tools were used in completing this practicum, you must explicitly declare their use. If you used AI, please include response the following three points:
1. ``Purpose``: A brief explanation of what you used generative AI for (e.g., brainstorming, proofreading, code debugging, or structuring ideas).
2. ``Sample Prompt``: At least one example of a prompt you submitted to the AI.
3. ``Refinement Process``: A short description of how you handled the prompt and evaluated the output (e.g., how you verified the facts, refined the wording, or altered the prompt to get better results).

In [2]:
# answer here



### Group Organisation and Contribution 

Since this assignment is a group project, your team must include a text addressing the following three points:
1. ``Task Organization``: Describe how your team organised the workflow to complete the entire notebook (e.g., who was responsible for which sections, how you collaborated, and how you merged your work).

2. ``Conflict Resolution``: Explain how you handled any discrepancies, disagreements, or differing opinions regarding the project's technical approach or final answers.

3. ``Percentage Contribution``: State the exact percentage of effort and contribution made by each member. The total sum across all three members must equal exactly 100% (e.g., Member A: 33.333%, Member B: 33.333%, Member C: 33.333%).


In [ ]:
# answer here

